In [1]:
import pandas as pd
import numpy as np
import os
import json
import re

In [2]:
def get_google_sheet(sheet_id: str, sheet_gid: str) -> pd.DataFrame:
	"""
	Downloads a specific sheet from a Google Sheet into a pandas DataFrame.

	Args:
		sheet_id: The ID of the Google Sheet.
		sheet_gid: The GID of the specific sheet to download.

	Returns:
		A pandas DataFrame containing the data from the specified sheet.
	"""
	url = f'https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv&gid={sheet_gid}'
	df = pd.read_csv(url)
	return df

# https://docs.google.com/spreadsheets/d/1sUHWAhl5ERNCGw1MvGRfEi-Ns-XYmFi027ZV5BRPeyw/edit?gid=1650661224#gid=1650661224 -> train
# https://docs.google.com/spreadsheets/d/1sUHWAhl5ERNCGw1MvGRfEi-Ns-XYmFi027ZV5BRPeyw/edit?gid=788722053#gid=788722053 -> test

## Without tags

In [ ]:
google_sheet_id = '1sUHWAhl5ERNCGw1MvGRfEi-Ns-XYmFi027ZV5BRPeyw'  # Replace with your actual Google Sheet ID
gid = '1650661224'  # Replace with the actual GID for the English sheet
try:
	df_correction = get_google_sheet(google_sheet_id, gid)
	print("Successfully loaded data from the specific sheet:")
except Exception as e:
	print(f"An error occurred: {e}")
	print("Please ensure the Google Sheet is shared correctly and the IDs are correct.")

In [21]:
df_correction

,sentence_id,element_order,input,targets,corrected_targets,is_target_corrected,is_double_checked,difficult_example,correction_class,corrected_targets_alt1,corrected_targets_alt2,corrected_targets_alt3,notes
0,0,aos,kamar saya ada kendala di ac tidak berfungsi o...,[A] ac [O] tidak berfungsi optimal [S] negativ...,[A] ac [O] tidak berfungsi optimal [S] negativ...,True,False,False,NaN,NaN,NaN,NaN,NaN
1,1,aos,tempatnya bagus . kolam renangnya bersih . [A]...,[A] tempatnya [O] bagus [S] positive\n[A] kola...,[A] tempatnya [O] bagus [S] positive\n[A] kola...,True,True,False,NaN,NaN,NaN,NaN,NaN
2,2,aos,"oke banget , tetapi ac nya tidak bisa diatur s...",[A] ac nya [O] tidak bisa diatur [S] negative\...,[A] ac nya [O] tidak bisa diatur suhu nya [S] ...,True,True,False,NaN,NaN,NaN,NaN,NaN
3,3,aos,keren . nyaman semuanya . [A] [O] [S],[A] semuanya [O] nyaman [S] positive\n[A] null...,[A] semuanya [O] nyaman [S] positive\n[A] null...,True,True,False,NaN,NaN,NaN,NaN,NaN
4,4,aos,"tidak dapat snack . setelah di keluhan , baru ...",[A] snack [O] tidak dapat [S] negative,[A] snack [O] tidak dapat [S] negative,True,True,False,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2477,2495,aos,wifi kurang joss . [A] [O] [S],[A] wifi [O] kurang joss [S] negative,NaN,True,True,False,NaN,NaN,NaN,NaN,NaN
2478,2496,aos,"kamar cukup bersih , hanya sempit , . [A] [O] [S]",[A] kamar [O] cukup bersih [S] positive\n[A] k...,NaN,True,True,False,NaN,NaN,NaN,NaN,NaN
2479,2497,aos,"nyaman , bersih , dan pelayananya sangat ramah...",[A] pelayananya [O] sangat ramah [S] positive\...,[A] pelayanannya [O] sangat ramah [S] positive...,True,True,False,NaN,[A] pelayanannya [O] sangat ramah [S] positive...,NaN,NaN,NaN
2480,2498,aos,sangat kecewa dengan kamar dan pelayanan stafn...,[A] kamar [O] sangat kecewa [S] negative\n[A] ...,NaN,True,True,False,NaN,NaN,NaN,NaN,NaN


In [22]:
# Fill empty values in corrected_targets with targets
df_correction['corrected_targets'] = df_correction['corrected_targets'].fillna(df_correction['targets'])

In [23]:
correction_data_arr = {
    'sentence_id': df_correction['sentence_id'].tolist(),
    'element_order': df_correction['element_order'].tolist(),
    'input': df_correction['input'].tolist(),
    'targets': df_correction['targets'].tolist(),
    'corrected_targets': df_correction['corrected_targets'].tolist(),
    'derived_from': [None] * len(df_correction), 
}

In [24]:
# Start idx for alternative solution train data -> 10000
# Start idx for alternative solution dev data -> 20000
# Start idx for alternative solution test data -> 30000

In [25]:
start_idx = 10000
for idx, row in df_correction.iterrows():
	for i in range(1, 4):
		alt_sol = row[f'corrected_targets_alt{i}']
		if pd.notna(alt_sol):
			correction_data_arr['sentence_id'].append(start_idx)
			correction_data_arr['element_order'].append(row['element_order'])
			correction_data_arr['input'].append(row['input'])
			correction_data_arr['targets'].append(row['targets'])
			correction_data_arr['corrected_targets'].append(alt_sol)
			correction_data_arr['derived_from'].append(row['sentence_id'])
			start_idx += 1


In [26]:
df_alt = pd.DataFrame(correction_data_arr)
df_alt['derived_from'] = df_alt['derived_from'].astype(pd.Int64Dtype())
df_alt

,sentence_id,element_order,input,targets,corrected_targets,derived_from
0,0,aos,kamar saya ada kendala di ac tidak berfungsi o...,[A] ac [O] tidak berfungsi optimal [S] negativ...,[A] ac [O] tidak berfungsi optimal [S] negativ...,<NA>
1,1,aos,tempatnya bagus . kolam renangnya bersih . [A]...,[A] tempatnya [O] bagus [S] positive\n[A] kola...,[A] tempatnya [O] bagus [S] positive\n[A] kola...,<NA>
2,2,aos,"oke banget , tetapi ac nya tidak bisa diatur s...",[A] ac nya [O] tidak bisa diatur [S] negative\...,[A] ac nya [O] tidak bisa diatur suhu nya [S] ...,<NA>
3,3,aos,keren . nyaman semuanya . [A] [O] [S],[A] semuanya [O] nyaman [S] positive\n[A] null...,[A] semuanya [O] nyaman [S] positive\n[A] null...,<NA>
4,4,aos,"tidak dapat snack . setelah di keluhan , baru ...",[A] snack [O] tidak dapat [S] negative,[A] snack [O] tidak dapat [S] negative,<NA>
...,...,...,...,...,...,...
3615,11133,aos,"terimakasih airy , pelayanan bagus , kamar san...",[A] pelayanan [O] bagus [S] positive\n[A] kama...,[A] pelayanan [O] bagus [S] positive\n[A] kama...,2487
3616,11134,aos,"ac tidak dingin , lembab , ada kecoak . [A] [O...",[A] ac [O] tidak dingin [S] negative\n[A] null...,[A] ac [O] tidak dingin [S] negative\n[A] null...,2489
3617,11135,aos,"suasana enak , hotel nuansa jawa banget atau k...",[A] suasana [O] enak [S] positive\n[A] kamar [...,[A] suasana [O] enak [S] positive\n[A] kamar [...,2490
3618,11136,aos,"hotel bagus , bersih , murah meriah . next tim...",[A] hotel [O] bagus [S] positive\n[A] hotel [O...,"[A] hotel [O] bagus , bersih , murah meriah . ...",2494


In [27]:
from typing import List, Dict
def parse_absa_string(text: str):
    """
    Parses a string formatted as "[A] aspect [O] opinion [S] sentiment" into a list of dictionaries.
    Each dictionary contains the tag as the key and the corresponding value.
    For example, "[A] [O] [S] [A] harga [O] terjangkau [S] positive [SSEP] [A] fasilitas [O] nyaman [S] positive" becomes:
    [{'A': 'harga', 'S': 'positive', 'O': 'terjangkau'},
    {'A': 'fasilitas', 'S': 'positive', 'O': 'nyaman'}].

    Args:
        text (str): ABSA string output to be parsed.

    Returns:
        List[Dict[str, str]]: List of dictionaries of parsed ABSA output.

    """
    pattern = r"\[(\w+)\]\s*([^[]+)"
    matches = re.findall(pattern, text)

    result = []
    current_dict = {}

    for tag, content in matches:
        if tag == "SSEP":  # Sentence separator -> Start a new dictionary
            result.append(current_dict)
            current_dict = {}
        else:
            current_dict[tag] = content.strip()

    if current_dict:  # Append the last sentence if it exists
        result.append(current_dict)

    return result



In [28]:
df_alt['corrected_targets_dict'] = df_alt['corrected_targets'].apply(lambda x: ' [SSEP] '.join(x.strip().split('\n'))).apply(lambda x: parse_absa_string(x))

In [31]:
# Get all permutations of ['A', 'O', 'S'] except the last one
from itertools import permutations
perm_order = ['aos', 'aso', 'sao', 'oas', 'osa']
perm_order = [list(x.upper()) for x in perm_order]
perm_order


[['A', 'O', 'S'],
 ['A', 'S', 'O'],
 ['S', 'A', 'O'],
 ['O', 'A', 'S'],
 ['O', 'S', 'A']]

In [32]:
from typing import Literal
def convert_to_absa_format(triplets: List[Dict[str, str]], order: List[Literal['A', 'O', 'S']]) -> str:
	"""
	Converts a list of dictionaries containing ABSA triplets into a formatted string.
	Each dictionary should contain keys 'A', 'O', and 'S' for Aspect, Opinion, and Sentiment respectively.
	The order of these elements in the output string is determined by the 'order' parameter.

	Args:
		triplets (List[Dict[str, str]]): List of dictionaries with ABSA triplet information.
	Returns:
		str: A formatted string representing the ABSA triplets.
	"""
	result = []
	for triplet in triplets:
		parts = []
		for key in order:
			if key in triplet:
				parts.append(f"[{key}] {triplet[key]}")
		result.append(" ".join(parts))
	return " [SSEP] ".join(result)

In [33]:
# Data Format for the augmented data
# "sentence_id": 0,
# "instance_id": 0,
# "task_elements": "aos",
# "input": "kamar saya ada kendala di ac tidak berfungsi optimal . dan juga wifi koneksi kurang stabil . [A] [O] [S]",
# "target": "[A] ac [O] tidak berfungsi optimal [S] negative [SSEP] [A] wifi koneksi [O] kurang stabil [S] negative",
# "element_order": "aos"

In [34]:
data_augmented = []
instance_id_mult = len(perm_order)
for idx, row in df_alt.iterrows():
	# Permuted orders
	for perm in perm_order:
		order_str = ''.join([x.lower() for x in perm])
		data_augmented.append({
			"sentence_id": row['sentence_id'],
			"instance_id": row['sentence_id'] * instance_id_mult + perm_order.index(perm),
			"task_elements": 'aos',
			"input": row['input'].strip().replace('[A] [O] [S]', f'[{perm[0]}] [{perm[1]}] [{perm[2]}]'),
			"target": convert_to_absa_format(row['corrected_targets_dict'], perm),
			"element_order": order_str,
			"derived_from": row['derived_from'] if pd.notna(row['derived_from']) else None
		})

In [35]:
df_augmented = pd.DataFrame(data_augmented)
df_augmented['derived_from'] = df_augmented['derived_from'].astype(pd.Int64Dtype())
df_augmented

,sentence_id,instance_id,task_elements,input,target,element_order,derived_from
0,0,0,aos,kamar saya ada kendala di ac tidak berfungsi o...,[A] ac [O] tidak berfungsi optimal [S] negativ...,aos,<NA>
1,0,1,aos,kamar saya ada kendala di ac tidak berfungsi o...,[A] ac [S] negative [O] tidak berfungsi optima...,aso,<NA>
2,0,2,aos,kamar saya ada kendala di ac tidak berfungsi o...,[S] negative [A] ac [O] tidak berfungsi optima...,sao,<NA>
3,0,3,aos,kamar saya ada kendala di ac tidak berfungsi o...,[O] tidak berfungsi optimal [A] ac [S] negativ...,oas,<NA>
4,0,4,aos,kamar saya ada kendala di ac tidak berfungsi o...,[O] tidak berfungsi optimal [S] negative [A] a...,osa,<NA>
...,...,...,...,...,...,...,...
18095,11137,55685,aos,"nyaman , bersih , dan pelayananya sangat ramah...",[A] pelayanannya [O] sangat ramah [S] positive...,aos,2497
18096,11137,55686,aos,"nyaman , bersih , dan pelayananya sangat ramah...",[A] pelayanannya [S] positive [O] sangat ramah...,aso,2497
18097,11137,55687,aos,"nyaman , bersih , dan pelayananya sangat ramah...",[S] positive [A] pelayanannya [O] sangat ramah...,sao,2497
18098,11137,55688,aos,"nyaman , bersih , dan pelayananya sangat ramah...",[O] sangat ramah [A] pelayanannya [S] positive...,oas,2497


In [36]:
# Write to json file
with open('../hotel_dataset/indo/corrected_all_alts/hotel_aste_train_augmented_noreasoning.json', 'w') as f:
	json.dump(data_augmented, f, indent=4)

## With tags

In [3]:
# https://docs.google.com/spreadsheets/d/1sUHWAhl5ERNCGw1MvGRfEi-Ns-XYmFi027ZV5BRPeyw/edit?gid=687074601#gid=687074601 -> train
# https://docs.google.com/spreadsheets/d/1sUHWAhl5ERNCGw1MvGRfEi-Ns-XYmFi027ZV5BRPeyw/edit?gid=906557225#gid=906557225 -> test

In [4]:
google_sheet_id = '1sUHWAhl5ERNCGw1MvGRfEi-Ns-XYmFi027ZV5BRPeyw'  # Replace with your actual Google Sheet ID
gid = '687074601'  # Replace with the actual GID for the English sheet
try:
	df_correction = get_google_sheet(google_sheet_id, gid)
	print("Successfully loaded data from the specific sheet:")
except Exception as e:
	print(f"An error occurred: {e}")
	print("Please ensure the Google Sheet is shared correctly and the IDs are correct.")

Successfully loaded data from the specific sheet:


In [5]:
# split_opinion, long_opinion, typo_present, typo_correction

In [6]:
df_correction

,sentence_id,element_order,input,targets,targets_tags,targets_tags_detail,corrected_targets,corrected_targets_tags,corrected_targets_tags_detail,corrected_targets_alt1,...,corrected_targets_alt1_tags_detail,corrected_targets_alt2,corrected_targets_alt2_tags,corrected_targets_alt2_tags_detail,corrected_targets_alt3,corrected_targets_alt3_tags,corrected_targets_alt3_tags_detail,num_solutions_before_correction,typo_and_opinion_mistakes,missing_tags_mistakes
0,0,aos,kamar saya ada kendala di ac tidak berfungsi o...,[A] ac [O] tidak berfungsi optimal [S] negativ...,NaN,NaN,[A] ac [O] tidak berfungsi optimal [S] negativ...,NaN,n_triplets=2;\nmax_opinion_len=3;\nmin_opinion...,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,False,False
1,1,aos,tempatnya bagus . kolam renangnya bersih . [A]...,[A] tempatnya [O] bagus [S] positive\n[A] kola...,NaN,NaN,[A] tempatnya [O] bagus [S] positive\n[A] kola...,NaN,n_triplets=2;\nmax_opinion_len=1;\nmin_opinion...,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,False,False
2,2,aos,"oke banget , tetapi ac nya tidak bisa diatur s...",[A] ac nya [O] tidak bisa diatur [S] negative\...,NaN,NaN,[A] ac nya [O] tidak bisa diatur suhu nya [S] ...,NaN,n_triplets=2;\nmax_opinion_len=5;\nmin_opinion...,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,False,False
3,3,aos,keren . nyaman semuanya . [A] [O] [S],[A] semuanya [O] nyaman [S] positive\n[A] null...,NaN,NaN,[A] semuanya [O] nyaman [S] positive\n[A] null...,NaN,n_triplets=2;\nmax_opinion_len=1;\nmin_opinion...,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,False,False
4,4,aos,"tidak dapat snack . setelah di keluhan , baru ...",[A] snack [O] tidak dapat [S] negative,NaN,NaN,[A] snack [O] tidak dapat [S] negative,NaN,n_triplets=1;\nmax_opinion_len=2;\nmin_opinion...,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2477,2495,aos,wifi kurang joss . [A] [O] [S],[A] wifi [O] kurang joss [S] negative,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,False,False
2478,2496,aos,"kamar cukup bersih , hanya sempit , . [A] [O] [S]",[A] kamar [O] cukup bersih [S] positive\n[A] k...,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,False,False
2479,2497,aos,"nyaman , bersih , dan pelayananya sangat ramah...",[A] pelayananya [O] sangat ramah [S] positive\...,NaN,NaN,[A] pelayanannya [O] sangat ramah [S] positive...,split_opinion,n_triplets=3;\nmax_opinion_len=2;\nmin_opinion...,[A] pelayanannya [O] sangat ramah [S] positive...,...,n_triplets=2;\nmax_opinion_len=3;\nmin_opinion...,NaN,NaN,NaN,NaN,NaN,NaN,2,False,False
2480,2498,aos,sangat kecewa dengan kamar dan pelayanan stafn...,[A] kamar [O] sangat kecewa [S] negative\n[A] ...,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,False,False


In [7]:
# Create a mask for rows where corrected_targets is null
corrected_targets_null_mask = df_correction['corrected_targets'].isna()

In [8]:
# Fill null values in corrected_targets with targets
df_correction['corrected_targets'] = df_correction['corrected_targets'].fillna(df_correction['targets'])

# Fill corrected_targets_tags with targets_tags only for rows where corrected_targets was originally null
df_correction.loc[corrected_targets_null_mask, 'corrected_targets_tags'] = df_correction.loc[corrected_targets_null_mask, 'targets_tags']
# df_correction.loc[corrected_targets_null_mask, 'corrected_targets_tags_detail'] = df_correction.loc[corrected_targets_null_mask, 'targets_tags_detail']

In [9]:
# Split and strip tags in all _tags columns
tags_columns = ['corrected_targets_tags', 'corrected_targets_alt1_tags', 'corrected_targets_alt2_tags', 'corrected_targets_alt3_tags']

for col in tags_columns:
	df_correction[col] = df_correction[col] \
		.apply(lambda x: x.split(',') if not pd.isna(x) else x) \
		.apply(lambda x: [tag.strip() for tag in x] if isinstance(x, list) else x)


In [10]:
df_correction

,sentence_id,element_order,input,targets,targets_tags,targets_tags_detail,corrected_targets,corrected_targets_tags,corrected_targets_tags_detail,corrected_targets_alt1,...,corrected_targets_alt1_tags_detail,corrected_targets_alt2,corrected_targets_alt2_tags,corrected_targets_alt2_tags_detail,corrected_targets_alt3,corrected_targets_alt3_tags,corrected_targets_alt3_tags_detail,num_solutions_before_correction,typo_and_opinion_mistakes,missing_tags_mistakes
0,0,aos,kamar saya ada kendala di ac tidak berfungsi o...,[A] ac [O] tidak berfungsi optimal [S] negativ...,NaN,NaN,[A] ac [O] tidak berfungsi optimal [S] negativ...,NaN,n_triplets=2;\nmax_opinion_len=3;\nmin_opinion...,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,False,False
1,1,aos,tempatnya bagus . kolam renangnya bersih . [A]...,[A] tempatnya [O] bagus [S] positive\n[A] kola...,NaN,NaN,[A] tempatnya [O] bagus [S] positive\n[A] kola...,NaN,n_triplets=2;\nmax_opinion_len=1;\nmin_opinion...,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,False,False
2,2,aos,"oke banget , tetapi ac nya tidak bisa diatur s...",[A] ac nya [O] tidak bisa diatur [S] negative\...,NaN,NaN,[A] ac nya [O] tidak bisa diatur suhu nya [S] ...,NaN,n_triplets=2;\nmax_opinion_len=5;\nmin_opinion...,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,False,False
3,3,aos,keren . nyaman semuanya . [A] [O] [S],[A] semuanya [O] nyaman [S] positive\n[A] null...,NaN,NaN,[A] semuanya [O] nyaman [S] positive\n[A] null...,NaN,n_triplets=2;\nmax_opinion_len=1;\nmin_opinion...,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,False,False
4,4,aos,"tidak dapat snack . setelah di keluhan , baru ...",[A] snack [O] tidak dapat [S] negative,NaN,NaN,[A] snack [O] tidak dapat [S] negative,NaN,n_triplets=1;\nmax_opinion_len=2;\nmin_opinion...,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2477,2495,aos,wifi kurang joss . [A] [O] [S],[A] wifi [O] kurang joss [S] negative,NaN,NaN,[A] wifi [O] kurang joss [S] negative,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,False,False
2478,2496,aos,"kamar cukup bersih , hanya sempit , . [A] [O] [S]",[A] kamar [O] cukup bersih [S] positive\n[A] k...,NaN,NaN,[A] kamar [O] cukup bersih [S] positive\n[A] k...,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,False,False
2479,2497,aos,"nyaman , bersih , dan pelayananya sangat ramah...",[A] pelayananya [O] sangat ramah [S] positive\...,NaN,NaN,[A] pelayanannya [O] sangat ramah [S] positive...,[split_opinion],n_triplets=3;\nmax_opinion_len=2;\nmin_opinion...,[A] pelayanannya [O] sangat ramah [S] positive...,...,n_triplets=2;\nmax_opinion_len=3;\nmin_opinion...,NaN,NaN,NaN,NaN,NaN,NaN,2,False,False
2480,2498,aos,sangat kecewa dengan kamar dan pelayanan stafn...,[A] kamar [O] sangat kecewa [S] negative\n[A] ...,NaN,NaN,[A] kamar [O] sangat kecewa [S] negative\n[A] ...,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,False,False


In [11]:
three_alt_indexes = df_correction.apply(lambda row: True if pd.notna(row['corrected_targets']) and pd.notna(row['corrected_targets_alt1']) and pd.notna(row['corrected_targets_alt2']) and pd.isna(row['corrected_targets_alt3']) else False, axis=1)
three_alt_indexes.sum()

np.int64(0)

In [12]:
three_alt_indexes = df_correction.apply(lambda row: True if pd.notna(row['corrected_targets']) and pd.notna(row['corrected_targets_alt1']) and pd.notna(row['corrected_targets_alt2']) and pd.isna(row['corrected_targets_alt3']) else False, axis=1)
# Make a new column named 'num_alternatives' to indicate how many alternative solutions are present
df_correction['num_solutions'] = df_correction.apply(lambda row: 1 + sum(pd.notna(row[f'corrected_targets_alt{i}']) for i in range(1, 4)), axis=1)


In [13]:
df_correction

,sentence_id,element_order,input,targets,targets_tags,targets_tags_detail,corrected_targets,corrected_targets_tags,corrected_targets_tags_detail,corrected_targets_alt1,...,corrected_targets_alt2,corrected_targets_alt2_tags,corrected_targets_alt2_tags_detail,corrected_targets_alt3,corrected_targets_alt3_tags,corrected_targets_alt3_tags_detail,num_solutions_before_correction,typo_and_opinion_mistakes,missing_tags_mistakes,num_solutions
0,0,aos,kamar saya ada kendala di ac tidak berfungsi o...,[A] ac [O] tidak berfungsi optimal [S] negativ...,NaN,NaN,[A] ac [O] tidak berfungsi optimal [S] negativ...,NaN,n_triplets=2;\nmax_opinion_len=3;\nmin_opinion...,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,1,False,False,1
1,1,aos,tempatnya bagus . kolam renangnya bersih . [A]...,[A] tempatnya [O] bagus [S] positive\n[A] kola...,NaN,NaN,[A] tempatnya [O] bagus [S] positive\n[A] kola...,NaN,n_triplets=2;\nmax_opinion_len=1;\nmin_opinion...,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,1,False,False,1
2,2,aos,"oke banget , tetapi ac nya tidak bisa diatur s...",[A] ac nya [O] tidak bisa diatur [S] negative\...,NaN,NaN,[A] ac nya [O] tidak bisa diatur suhu nya [S] ...,NaN,n_triplets=2;\nmax_opinion_len=5;\nmin_opinion...,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,1,False,False,1
3,3,aos,keren . nyaman semuanya . [A] [O] [S],[A] semuanya [O] nyaman [S] positive\n[A] null...,NaN,NaN,[A] semuanya [O] nyaman [S] positive\n[A] null...,NaN,n_triplets=2;\nmax_opinion_len=1;\nmin_opinion...,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,1,False,False,1
4,4,aos,"tidak dapat snack . setelah di keluhan , baru ...",[A] snack [O] tidak dapat [S] negative,NaN,NaN,[A] snack [O] tidak dapat [S] negative,NaN,n_triplets=1;\nmax_opinion_len=2;\nmin_opinion...,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,1,False,False,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2477,2495,aos,wifi kurang joss . [A] [O] [S],[A] wifi [O] kurang joss [S] negative,NaN,NaN,[A] wifi [O] kurang joss [S] negative,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,1,False,False,1
2478,2496,aos,"kamar cukup bersih , hanya sempit , . [A] [O] [S]",[A] kamar [O] cukup bersih [S] positive\n[A] k...,NaN,NaN,[A] kamar [O] cukup bersih [S] positive\n[A] k...,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,1,False,False,1
2479,2497,aos,"nyaman , bersih , dan pelayananya sangat ramah...",[A] pelayananya [O] sangat ramah [S] positive\...,NaN,NaN,[A] pelayanannya [O] sangat ramah [S] positive...,[split_opinion],n_triplets=3;\nmax_opinion_len=2;\nmin_opinion...,[A] pelayanannya [O] sangat ramah [S] positive...,...,NaN,NaN,NaN,NaN,NaN,NaN,2,False,False,2
2480,2498,aos,sangat kecewa dengan kamar dan pelayanan stafn...,[A] kamar [O] sangat kecewa [S] negative\n[A] ...,NaN,NaN,[A] kamar [O] sangat kecewa [S] negative\n[A] ...,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,1,False,False,1


In [14]:
# df_correction.loc[:, 'num_solutions'].to_csv('num_solutions.csv', index=False)

In [15]:
# correction_data_arr = {
#     'sentence_id': df_correction['sentence_id'].tolist(),
#     'element_order': df_correction['element_order'].tolist(),
#     'input': df_correction['input'].tolist(),
#     'targets': df_correction['targets'].tolist(),
#     'corrected_targets': df_correction['corrected_targets'].tolist(),
#     'derived_from': [None] * len(df_correction), 
# }

In [16]:
typo_and_opinion_mistakes = []
missing_tags_mistakes = []

def select_by_tag(row, tags, not_complete_mode=False):
	global typo_and_opinion_mistakes, missing_tags_mistakes
	tags_columns = ['corrected_targets_tags', 'corrected_targets_alt1_tags', 'corrected_targets_alt2_tags', 'corrected_targets_alt3_tags']
	
	if row['num_solutions'] == 1: # Only one solution available
		return pd.Series({
			'sentence_id': row['sentence_id'], 
			'element_order': row['element_order'],
			'input': row['input'],
			'targets': row['targets'],
			'corrected_targets': row['corrected_targets'], 
			'num_solutions': row['num_solutions'],
			'tags': row['corrected_targets_tags'] if pd.notna(row['corrected_targets_tags']) else []
		})
	elif row['num_solutions'] == 2: # Choose between two solutions based on tags
		candidate_solution = []
		for tag_column in tags_columns:
			if pd.notna(row[tag_column.replace('_tags', '')]):
				candidate_solution.append({
					'sentence_id': row['sentence_id'],
					'element_order': row['element_order'],
					'input': row['input'],
					'targets': row['targets'],
					'target': row[tag_column.replace('_tags', '')], # String of the solution
					'tag': row[tag_column] # List of tags
				})
		
		# Check if there is any duplicate in the tags of the candidate solution
		tags_candidate = []
		for d in candidate_solution:
			# Ensure all tags are lists
			if not isinstance(d['tag'], list):
				print(f"Tags are not a list in row with sentence_id {row['sentence_id']}")
				missing_tags_mistakes.append(row['sentence_id'])
				return pd.Series({'sentence_id': None, 'element_order': None, 'input': None, 'targets': None, 'corrected_targets': None, 'num_solutions': None, 'tags': None})
			tags_candidate.extend(d['tag'])

		# Check for duplicates
		if len(tags_candidate) != len(set(tags_candidate)):
			print(f"Duplicate tags found in row with sentence_id {row['sentence_id']}")
			typo_and_opinion_mistakes.append(row['sentence_id'])
			
			if not_complete_mode: # If not complete mode, return None
				for d in candidate_solution:
					valid = True
					for tag in tags:
						if tag not in d['tag']:
							valid = False
							break
					if valid:
						return pd.Series({
							'sentence_id': d['sentence_id'],
							'element_order': d['element_order'],
							'input': d['input'],
							'targets': d['targets'],
							'corrected_targets': d['target'],
							'num_solutions': row['num_solutions'],
							'tags': d['tag']
						})

			return pd.Series({'sentence_id': None, 'element_order': None, 'input': None, 'targets': None, 'corrected_targets': None, 'num_solutions': None, 'tags': None})
		
		# Check if both candidate solutions have the same kinds of tag i.e. both have either 'typo' or 'opinion' but not 'typo' and 'opinion' at the same time
		typo_present = any(['typo' in tag_candidate for tag_candidate in tags_candidate])
		opinion_present = any(['opinion' in tag_candidate for tag_candidate in tags_candidate])
		if typo_present and opinion_present:
			print(f"Both 'typo' and 'opinion' tags found in row with sentence_id {row['sentence_id']}")
			typo_and_opinion_mistakes.append(row['sentence_id'])

			if not_complete_mode: # If not complete mode, return None
				for d in candidate_solution:
					valid = True
					for tag in tags:
						if tag not in d['tag']:
							valid = False
							break
					if valid:
						return pd.Series({
							'sentence_id': d['sentence_id'],
							'element_order': d['element_order'],
							'input': d['input'],
							'targets': d['targets'],
							'corrected_targets': d['target'],
							'num_solutions': row['num_solutions'],
							'tags': d['tag']
						})

			return pd.Series({'sentence_id': None, 'element_order': None, 'input': None, 'targets': None, 'corrected_targets': None, 'num_solutions': None, 'tags': None})
		
		# Select the solution that matches the provided tags
		for d in candidate_solution:
			for tag in tags:
				if typo_present:
					if 'typo' not in tag:
						continue
				if opinion_present:
					if 'opinion' not in tag:
						continue
				if tag in d['tag']:
					return pd.Series({
						'sentence_id': d['sentence_id'], 
						'element_order': d['element_order'],
						'input': d['input'],
						'targets': d['targets'],
						'corrected_targets': d['target'], 
						'num_solutions': row['num_solutions'],
						'tags': d['tag']
					})
			
		print(f"No matching tags found in row with sentence_id {row['sentence_id']} for num_solutions == 2")
		return pd.Series({'sentence_id': None, 'element_order': None, 'input': None, 'targets': None, 'corrected_targets': None, 'num_solutions': None, 'tags': None})

	elif row['num_solutions'] == 4: # Choose between four solutions based on tags
		candidate_solution = []
		for tag_column in tags_columns:
			if pd.notna(row[tag_column.replace('_tags', '')]):
				candidate_solution.append({
					'sentence_id': row['sentence_id'],
					'element_order': row['element_order'],
					'input': row['input'],
					'targets': row['targets'],
					'target': row[tag_column.replace('_tags', '')],
					'tag': row[tag_column]
				})
		
		# Select the solution that matches the provided tags
		for d in candidate_solution:
			if all(tag in d['tag'] for tag in tags):
				return pd.Series({
					'sentence_id': d['sentence_id'], 
					'element_order': d['element_order'],
					'input': d['input'],
					'targets': d['targets'],
					'corrected_targets': d['target'], 
					'num_solutions': row['num_solutions'],
					'tags': d['tag']
				})
			
		print(f"No matching tags found in row with sentence_id {row['sentence_id']} for num_solutions == 4")
		print(f"Available tags: {', '.join([tag for d in candidate_solution for tag in d['tag']])}")
		return pd.Series({'sentence_id': None, 'element_order': None, 'input': None, 'targets': None, 'corrected_targets': None, 'num_solutions': None, 'tags': None})

# Use result_type='expand' to create separate columns
final_corrected_targets = df_correction.apply(lambda row: select_by_tag(row, ['split_opinion', 'typo_corrected'], not_complete_mode=True), axis=1, result_type='expand')

Duplicate tags found in row with sentence_id 17
Duplicate tags found in row with sentence_id 738
Duplicate tags found in row with sentence_id 870
Duplicate tags found in row with sentence_id 886
Duplicate tags found in row with sentence_id 1177
Duplicate tags found in row with sentence_id 1285
Duplicate tags found in row with sentence_id 1324
Duplicate tags found in row with sentence_id 1374
Duplicate tags found in row with sentence_id 1388
Duplicate tags found in row with sentence_id 1519
Duplicate tags found in row with sentence_id 1724
Duplicate tags found in row with sentence_id 1990
Duplicate tags found in row with sentence_id 2042
Both 'typo' and 'opinion' tags found in row with sentence_id 2050
Both 'typo' and 'opinion' tags found in row with sentence_id 2068
Duplicate tags found in row with sentence_id 2075
Duplicate tags found in row with sentence_id 2285


In [17]:
final_corrected_targets

,sentence_id,element_order,input,targets,corrected_targets,num_solutions,tags
0,0,aos,kamar saya ada kendala di ac tidak berfungsi o...,[A] ac [O] tidak berfungsi optimal [S] negativ...,[A] ac [O] tidak berfungsi optimal [S] negativ...,1,[]
1,1,aos,tempatnya bagus . kolam renangnya bersih . [A]...,[A] tempatnya [O] bagus [S] positive\n[A] kola...,[A] tempatnya [O] bagus [S] positive\n[A] kola...,1,[]
2,2,aos,"oke banget , tetapi ac nya tidak bisa diatur s...",[A] ac nya [O] tidak bisa diatur [S] negative\...,[A] ac nya [O] tidak bisa diatur suhu nya [S] ...,1,[]
3,3,aos,keren . nyaman semuanya . [A] [O] [S],[A] semuanya [O] nyaman [S] positive\n[A] null...,[A] semuanya [O] nyaman [S] positive\n[A] null...,1,[]
4,4,aos,"tidak dapat snack . setelah di keluhan , baru ...",[A] snack [O] tidak dapat [S] negative,[A] snack [O] tidak dapat [S] negative,1,[]
...,...,...,...,...,...,...,...
2477,2495,aos,wifi kurang joss . [A] [O] [S],[A] wifi [O] kurang joss [S] negative,[A] wifi [O] kurang joss [S] negative,1,[]
2478,2496,aos,"kamar cukup bersih , hanya sempit , . [A] [O] [S]",[A] kamar [O] cukup bersih [S] positive\n[A] k...,[A] kamar [O] cukup bersih [S] positive\n[A] k...,1,[]
2479,2497,aos,"nyaman , bersih , dan pelayananya sangat ramah...",[A] pelayananya [O] sangat ramah [S] positive\...,[A] pelayanannya [O] sangat ramah [S] positive...,2,[split_opinion]
2480,2498,aos,sangat kecewa dengan kamar dan pelayanan stafn...,[A] kamar [O] sangat kecewa [S] negative\n[A] ...,[A] kamar [O] sangat kecewa [S] negative\n[A] ...,1,[]


In [18]:
final_corrected_targets.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2482 entries, 0 to 2481
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   sentence_id        2482 non-null   int64 
 1   element_order      2482 non-null   object
 2   input              2482 non-null   object
 3   targets            2482 non-null   object
 4   corrected_targets  2482 non-null   object
 5   num_solutions      2482 non-null   int64 
 6   tags               2482 non-null   object
dtypes: int64(2), object(5)
memory usage: 135.9+ KB


In [19]:
# Create true/false column for typo_and_opinion_mistakes and missing_tags_mistakes
df_correction['typo_and_opinion_mistakes'] = df_correction['sentence_id'].apply(lambda x: x in typo_and_opinion_mistakes)
df_correction['missing_tags_mistakes'] = df_correction['sentence_id'].apply(lambda x: x in missing_tags_mistakes)
df_correction[['num_solutions', 'typo_and_opinion_mistakes', 'missing_tags_mistakes']].to_csv('mistakes_test.csv', index=False)

In [20]:
from typing import List, Dict
def parse_absa_string(text: str):
    """
    Parses a string formatted as "[A] aspect [O] opinion [S] sentiment" into a list of dictionaries.
    Each dictionary contains the tag as the key and the corresponding value.
    For example, "[A] [O] [S] [A] harga [O] terjangkau [S] positive [SSEP] [A] fasilitas [O] nyaman [S] positive" becomes:
    [{'A': 'harga', 'S': 'positive', 'O': 'terjangkau'},
    {'A': 'fasilitas', 'S': 'positive', 'O': 'nyaman'}].

    Args:
        text (str): ABSA string output to be parsed.

    Returns:
        List[Dict[str, str]]: List of dictionaries of parsed ABSA output.

    """
    pattern = r"\[(\w+)\]\s*([^[]+)"
    matches = re.findall(pattern, text)

    result = []
    current_dict = {}

    for tag, content in matches:
        if tag == "SSEP":  # Sentence separator -> Start a new dictionary
            result.append(current_dict)
            current_dict = {}
        else:
            current_dict[tag] = content.strip()

    if current_dict:  # Append the last sentence if it exists
        result.append(current_dict)

    return result

final_corrected_targets['corrected_targets_dict'] = final_corrected_targets['corrected_targets'].apply(lambda x: ' [SSEP] '.join(x.strip().split('\n'))).apply(lambda x: parse_absa_string(x))

In [21]:
# Get all permutations of ['A', 'O', 'S'] except the last one
from itertools import permutations
perm_order = ['aos', 'aso', 'sao', 'oas', 'osa']
perm_order = [list(x.upper()) for x in perm_order]
perm_order


[['A', 'O', 'S'],
 ['A', 'S', 'O'],
 ['S', 'A', 'O'],
 ['O', 'A', 'S'],
 ['O', 'S', 'A']]

In [22]:
from typing import Literal
def convert_to_absa_format(triplets: List[Dict[str, str]], order: List[Literal['A', 'O', 'S']]) -> str:
	"""
	Converts a list of dictionaries containing ABSA triplets into a formatted string.
	Each dictionary should contain keys 'A', 'O', and 'S' for Aspect, Opinion, and Sentiment respectively.
	The order of these elements in the output string is determined by the 'order' parameter.

	Args:
		triplets (List[Dict[str, str]]): List of dictionaries with ABSA triplet information.
	Returns:
		str: A formatted string representing the ABSA triplets.
	"""
	result = []
	for triplet in triplets:
		parts = []
		for key in order:
			if key in triplet:
				parts.append(f"[{key}] {triplet[key]}")
		result.append(" ".join(parts))
	return " [SSEP] ".join(result)

In [23]:
# Data Format for the augmented data
# "sentence_id": 0,
# "instance_id": 0,
# "task_elements": "aos",
# "input": "kamar saya ada kendala di ac tidak berfungsi optimal . dan juga wifi koneksi kurang stabil . [A] [O] [S]",
# "target": "[A] ac [O] tidak berfungsi optimal [S] negative [SSEP] [A] wifi koneksi [O] kurang stabil [S] negative",
# "element_order": "aos"

In [24]:
data_augmented = []
instance_id_mult = len(perm_order)
for idx, row in final_corrected_targets.iterrows():
	# Permuted orders
	for perm in perm_order:
		order_str = ''.join([x.lower() for x in perm])
		data_augmented.append({
			"sentence_id": row['sentence_id'],
			"instance_id": row['sentence_id'] * instance_id_mult + perm_order.index(perm),
			"task_elements": 'aos',
			"input": row['input'].strip().replace('[A] [O] [S]', f'[{perm[0]}] [{perm[1]}] [{perm[2]}]'),
			"target": convert_to_absa_format(row['corrected_targets_dict'], perm),
			"element_order": order_str,
		})

In [25]:
df_augmented = pd.DataFrame(data_augmented)
df_augmented

,sentence_id,instance_id,task_elements,input,target,element_order
0,0,0,aos,kamar saya ada kendala di ac tidak berfungsi o...,[A] ac [O] tidak berfungsi optimal [S] negativ...,aos
1,0,1,aos,kamar saya ada kendala di ac tidak berfungsi o...,[A] ac [S] negative [O] tidak berfungsi optima...,aso
2,0,2,aos,kamar saya ada kendala di ac tidak berfungsi o...,[S] negative [A] ac [O] tidak berfungsi optima...,sao
3,0,3,aos,kamar saya ada kendala di ac tidak berfungsi o...,[O] tidak berfungsi optimal [A] ac [S] negativ...,oas
4,0,4,aos,kamar saya ada kendala di ac tidak berfungsi o...,[O] tidak berfungsi optimal [S] negative [A] a...,osa
...,...,...,...,...,...,...
12405,2499,12495,aos,"sayang , air panasnya tidak terlalu panas , ja...","[A] air panasnya [O] tidak terlalu panas , jad...",aos
12406,2499,12496,aos,"sayang , air panasnya tidak terlalu panas , ja...",[A] air panasnya [S] negative [O] tidak terlal...,aso
12407,2499,12497,aos,"sayang , air panasnya tidak terlalu panas , ja...",[S] negative [A] air panasnya [O] tidak terlal...,sao
12408,2499,12498,aos,"sayang , air panasnya tidak terlalu panas , ja...","[O] tidak terlalu panas , jadi kalau mandi ked...",oas


In [26]:
# Write to json file
os.makedirs('../hotel_dataset/indo/corrected_splitopinion_typocorrected', exist_ok=True)
with open('../hotel_dataset/indo/corrected_splitopinion_typocorrected/hotel_aste_train_augmented_noreasoning.json', 'w') as f:
	json.dump(data_augmented, f, indent=4)